# 02 - Main ablation on GPT-4o-mini

Full 5 (prompt structure) x 2 (category hint) factorial on all 360 samples.
CLEAN and HINTED differ only by one identical category-naming clause.

In [1]:
# --- environment ---
!pip -q install openai
from google.colab import drive; drive.mount('/content/drive')

import sys
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/LLM_Security_Paper')   # SARD corpus root
WORK = ROOT / 'revision_2026'                              # outputs live here
WORK.mkdir(exist_ok=True)

sys.path.insert(0, str(WORK))          # vulnbench.py lives in WORK
import vulnbench as vb

BENCH = pd.read_csv(WORK / 'benchmark_360_metadata.csv')
print(len(BENCH), 'samples |', BENCH.true_label.value_counts().to_dict())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
360 samples | {'Safe': 180, 'Vulnerable': 180}


In [2]:
# ---- 預檢 A：檢視實際送進模型的內容（不花錢）----
import re, importlib
importlib.reload(vb)                      # 確保用到新版模組

row = BENCH.iloc[0]
code = vb.strip_header((ROOT / row['rel_path']).read_text(encoding='utf-8', errors='replace'))
instr = vb.build_prompt('C', 'hinted', row['cwe'])
print('=== SYSTEM ===');  print(vb.SYSTEM_MESSAGE)
print('\n=== USER ===');  print(f'{instr}\n\n```php\n{code}\n```')

# ---- 預檢 B：全 360 檔掃描洩漏字串 ----
LEAK = re.compile(r'(un)?safe sample|Copyright|sanitize\s*:|input\s*:\s*reads', re.I)
bad = [r['rel_path'] for _, r in BENCH.iterrows()
       if LEAK.search(vb.strip_header((ROOT / r['rel_path']).read_text(encoding='utf-8', errors='replace')))]
print(f'\n洩漏殘留檔案數: {len(bad)}')
assert not bad, bad[:5]
print('✓ 360 個樣本皆無標籤洩漏')

=== SYSTEM ===
You must respond strictly in JSON format with exactly three keys:
1. "prediction": String, MUST be either "Vulnerable" or "Safe".
2. "cwe_type": String, the identified CWE identifier (for example, "CWE-XX") or "None" if safe.
3. "confidence": Integer from 0 to 100.

=== USER ===
Analyze the following PHP code and determine whether it contains a security vulnerability. Apply taint analysis: trace data flow from untrusted user inputs (sources such as $_GET, $_POST, $_REQUEST, $_COOKIE, $_SESSION) to security-sensitive operations (sinks), and determine whether any unsanitized tainted data reaches a sink. The vulnerability class of interest is OS command injection (CWE-78). Return JSON.

```php
<?php
$tainted = $_GET['UserData'];

$sanitized = filter_var($tainted, FILTER_SANITIZE_MAGIC_QUOTES);
  $tainted = $sanitized ;
      

$query = "ls ' $tainted '";

$ret = system($query);


?>
```

洩漏殘留檔案數: 0
✓ 360 個樣本皆無標籤洩漏


In [3]:
import os, getpass
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
from openai import OpenAI
client = OpenAI()

OpenAI API key: ··········


In [4]:
# ---- 預檢 C：洩漏探針，24 次呼叫約 $0.006 ----
probe = (BENCH.groupby('true_label', group_keys=False)
              .apply(lambda d: d.sample(12, random_state=1)))
r = vb.run_experiment(probe, ROOT, [('A','clean')], 'gpt-3.5-turbo-0125',
                      WORK/'probe.csv', temperature=0.1, client=client)
print(pd.crosstab(r.true_label, r.prediction))

/tmp/ipykernel_1512/2480322757.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: d.sample(12, random_state=1)))


24 calls to make with gpt-3.5-turbo-0125
  24/24  |  tokens in/out 6049/696  |  running cost $0.004

done — 24 rows in /content/drive/MyDrive/LLM_Security_Paper/revision_2026/probe.csv
this session cost ≈ $0.004
prediction  Vulnerable
true_label            
Safe                12
Vulnerable          12


In [5]:
# 5 prompt structures x 2 hint conditions = 10 conditions
CONDITIONS = [(v, h) for v in 'ABCDE' for h in ('clean', 'hinted')]
print(CONDITIONS)
print('calls:', len(BENCH) * len(CONDITIONS))

[('A', 'clean'), ('A', 'hinted'), ('B', 'clean'), ('B', 'hinted'), ('C', 'clean'), ('C', 'hinted'), ('D', 'clean'), ('D', 'hinted'), ('E', 'clean'), ('E', 'hinted')]
calls: 3600


In [6]:
res = vb.run_experiment(
    bench        = BENCH,
    dataset_root = ROOT,
    conditions   = CONDITIONS,
    model        = 'gpt-4o-mini',
    temperature  = 0.1,
    out_csv      = WORK / 'main_4omini.csv',
    client       = client,
)

3600 calls to make with gpt-4o-mini
  50/3600  |  tokens in/out 9291/1500  |  running cost $0.002
  100/3600  |  tokens in/out 19068/3000  |  running cost $0.005
  150/3600  |  tokens in/out 32329/4500  |  running cost $0.008
  200/3600  |  tokens in/out 46910/6000  |  running cost $0.011
  250/3600  |  tokens in/out 60945/7500  |  running cost $0.014
  300/3600  |  tokens in/out 70099/9000  |  running cost $0.016
  350/3600  |  tokens in/out 80272/10500  |  running cost $0.018
  400/3600  |  tokens in/out 90343/12000  |  running cost $0.021
  450/3600  |  tokens in/out 100736/13495  |  running cost $0.023
  500/3600  |  tokens in/out 113612/14990  |  running cost $0.026
  550/3600  |  tokens in/out 129311/16490  |  running cost $0.029
  600/3600  |  tokens in/out 145172/17990  |  running cost $0.033
  650/3600  |  tokens in/out 155178/19490  |  running cost $0.035
  700/3600  |  tokens in/out 165571/20990  |  running cost $0.037
  750/3600  |  tokens in/out 175835/22490  |  running co

In [8]:
ok = res[res.prediction.notna()]
print(pd.crosstab(ok.condition, ok.prediction))
print()
print('unparsable calls:', res.prediction.isna().sum())

prediction  Safe  Vulnerable
condition                   
A_clean        0         360
A_hinted       2         358
B_clean        0         360
B_hinted       1         359
C_clean        3         357
C_hinted       5         355
D_clean        1         359
D_hinted       4         356
E_clean        2         358
E_hinted       0         360

unparsable calls: 0
